In [1]:
# Set-up
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

CONFIG_ROOT = Path.cwd().parents[2]/"UZH_Decarb_Project"
sys.path.append(str(CONFIG_ROOT))
import config_nicolas #load my version of config

CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
# from openpyxl import Workbook
# from openpyxl.styles import Font, Alignment
# import os

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load data
groups = pd.read_csv( #TODO explore data
    config_nicolas.PROCESSED_DATA/"individual_processed_4.csv", # Load in most recent version of 
    keep_default_na = False,  #Keep "None" as a string, not NaN
    na_values = [""] #Only treat empty strings as NaN
)

In [3]:
# Acquire counts of each type of faculty
faculty_counts = groups["faculty"].value_counts()
# print(faculty_counts)

# Histogram of lab groups
ax = faculty_counts.plot(kind = 'bar', color = 'cornflowerblue') #create object
plt.xlabel('faculty')
plt.ylabel('Count')
plt.title('Number of groups per faculty')
ax.set_xticklabels(["MNF", "MeF", "Both"]) #Replace with shorter names
plt.xticks(rotation = 0) #Orient horizontally
ax.set_xlabel("Faculty") #Correct figure x-axis label
plt.savefig("FacultyGroupCounts.png")
plt.close()


In [4]:
#Gather relevant waste values
waste_values = groups[["waste_clinical", "waste_general", "waste_recycle"]]
# waste_categories = ["None", "0-3kg", "3-6kg", "6-10kg", "10-12kg", "> 12kg", "Unknown"]
sensitive_categories = ["< 3kg", "3-10kg", "> 10kg", "Unknown"]

for i, col in enumerate(waste_values.columns):
    #Convert to suitable title format
    x_axis_title = col.title().replace("_", " ")

    #Change values to ensure sensitivity (verified through previous version) (the wrong way)
    waste_values.loc[waste_values[col] == "None", col] = "< 3kg"
    waste_values.loc[waste_values[col] == "0-3kg", col] = "< 3kg"
    waste_values.loc[waste_values[col] == "3-6kg", col] = "3-10kg"
    waste_values.loc[waste_values[col] == "6-10kg", col] = "3-10kg"
    waste_values.loc[waste_values[col] == "10-12kg", col] = "> 10kg"
    waste_values.loc[waste_values[col] == "> 12kg", col] = "> 10kg"
    # print(waste_values)
    # #Reorder to ordinal (https://stackoverflow.com/questions/73411349/sort-histogram-columns-order-with-seaborn)
    # waste_values[col] = pd.Categorical(values = waste_values[col], categories = sensitive_categories)
    # waste_values.sort_values([col], inplace = True) #ensures ordinality for plot
    
    # #Plot data
    # # sns.set_style("whitegrid")
    # plt.figure(i)
    # # plt.tight_layout()
    # ax = sns.histplot(waste_values[col], stat = "percent", color = "slategray")
    # #remove box
    # # ax.spines["top"].set_visible(False)
    # # ax.spines["right"].set_visible(False)
    # ax.set_xlabel(x_axis_title)

    # #Save figure
    # fig = ax.get_figure()
    # fig.savefig(col + ".png")
#Pivot longer for plotting purposes
waste_values = pd.melt(waste_values, value_vars = ["waste_general", "waste_clinical", "waste_recycle"], var_name = "Waste Type", value_name = "Waste Mass")
#Raname Variables for visual
waste_values.loc[waste_values["Waste Type"] == "waste_general", "Waste Type"] = "General"
waste_values.loc[waste_values["Waste Type"] == "waste_clinical", "Waste Type"] = "Clinical"
waste_values.loc[waste_values["Waste Type"] == "waste_recycle", "Waste Type"] = "Recycle"
#Reorder to ordinal (https://stackoverflow.com/questions/73411349/sort-histogram-columns-order-with-seaborn)
waste_values["Waste Mass"] = pd.Categorical(values = waste_values["Waste Mass"], categories = sensitive_categories)
waste_values.sort_values(["Waste Mass"], inplace = True) #ensures ordinality for plot
#Create facet grid of plots
waste_plot = sns.FacetGrid(waste_values,
                           col = "Waste Type", #loop over this variable
                           col_order = ["General", "Clinical", "Recycle"], #match survey order
                           aspect = 1.5) #wider for readability
waste_plot.map(sns.histplot, "Waste Mass", stat = "percent", color = "slategray")
#Save plot
waste_plot.savefig("Waste Plot.png")
plt.close()



In [ ]:
#Look into comments of the other machine types, skipping microwave and coffee machine
strange_equip = groups[["pcr_co", "ice_co", "centrifuge_co", "animal_co", "nonco2_incubator_co", "4c_room_co", "minus_20c_room_co", "other_co"]]
strange_equip = strange_equip.dropna(how = "all") #only interested in valid entries

#Extract only non NaN values for examination
strange_equip = strange_equip.stack().values

#Save equipment as a txt file for examination
with open("Strange_Equipment.txt", "w") as output:
    output.write(str(strange_equip))